# GraviLearn - Machine learning pipelines

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil*

In [ ]:
#r "../src/GraviNum/bin/Release/net10.0/Gravicode.Science.GraviNum.dll"
#r "../src/GraviFrame/bin/Release/net10.0/Gravicode.Science.GraviFrame.dll"
#r "../src/GraviLearn/bin/Release/net10.0/Gravicode.Science.GraviLearn.dll"
#r "nuget: ScottPlot, 5.1.59"

using Gravicode.Science.GraviLearn;
using Gravicode.Science.GraviLearn.Clustering;
using Gravicode.Science.GraviLearn.Decomposition;
using Gravicode.Science.GraviLearn.ModelSelection;
using Gravicode.Science.GraviLearn.Preprocessing;
using Gravicode.Science.GraviLearn.Trees;
using Gravicode.Science.GraviNum;

var iris = Datasets.LoadIris();
Console.WriteLine($"{iris.SampleCount} samples, {iris.FeatureCount} features, {iris.TargetNames.Count} classes");

## Train a random forest

In [ ]:
var split = Selection.Split(iris.Features, iris.Target, testSize: 0.3, seed: 42, stratify: true);

var forest = new RandomForestClassifier(nTrees: 200, seed: 42);
forest.Fit(split.TrainX, split.TrainY);

Console.WriteLine($"test accuracy       : {forest.Score(split.TestX, split.TestY):P2}");
Console.WriteLine($"out-of-bag accuracy : {forest.OutOfBagScore:P2}");

## Classification report

In [ ]:
var predictions = forest.Predict(split.TestX);
Console.WriteLine(Metrics.ClassificationReport(split.TestY, predictions, iris.LabelNames));

## Confusion matrix

In [ ]:
var digits = Datasets.LoadDigits();
var digitSplit = Selection.Split(digits.Features, digits.Target, testSize: 0.3, seed: 42, stratify: true);

var pipeline = new Pipeline()
    .Add(new StandardScaler())
    .Add(new PCA(components: 30))
    .Add(new Gravicode.Science.GraviLearn.Neighbors.KNearestNeighborsClassifier(k: 3));
pipeline.Fit(digitSplit.TrainX, digitSplit.TrainY);

var confusion = Metrics.ConfusionMatrix(digitSplit.TestY, pipeline.Predict(digitSplit.TestX));

var plot = new ScottPlot.Plot();
var heatmap = plot.Add.Heatmap(confusion.To2DArray());
heatmap.Colormap = new ScottPlot.Colormaps.Viridis();
plot.Add.ColorBar(heatmap);
plot.Title($"digits confusion matrix - {pipeline.Score(digitSplit.TestX, digitSplit.TestY):P1} accuracy");
plot.XLabel("predicted"); plot.YLabel("true");
plot.GetImageHtml(800, 700)

## Compare models with cross-validation

In [ ]:
var candidates = new (string, Func<IEstimator>)[]
{
    ("LogisticRegression", () => new Gravicode.Science.GraviLearn.Linear.LogisticRegression(0.3, 1500)),
    ("DecisionTree", () => new DecisionTree(maxDepth: 4)),
    ("RandomForest", () => new RandomForestClassifier(nTrees: 100, seed: 42)),
    ("kNN", () => new Gravicode.Science.GraviLearn.Neighbors.KNearestNeighborsClassifier(k: 5)),
};

foreach (var (name, factory) in candidates)
{
    var result = Selection.CrossValidate(factory, iris.Features, iris.Target, folds: 5, stratified: true, seed: 42);
    Console.WriteLine($"{name,-22}{result}");
}

## Principal components and clustering

In [ ]:
var scaled = new StandardScaler().FitTransform(iris.Features);
var pca = new PrincipalComponentAnalysis(2);
var projected = pca.FitTransform(iris.Features);

Console.WriteLine($"variance explained: PC1 {pca.ExplainedVarianceRatio.At(0):P2}, PC2 {pca.ExplainedVarianceRatio.At(1):P2}");

var plot2 = new ScottPlot.Plot();
foreach (var group in Enumerable.Range(0, iris.SampleCount).GroupBy(i => (int)iris.Target.At(i)))
{
    var xs = group.Select(i => projected[i, 0]).ToArray();
    var ys = group.Select(i => projected[i, 1]).ToArray();
    var scatter = plot2.Add.ScatterPoints(xs, ys);
    scatter.LegendText = iris.TargetNames[group.Key];
    scatter.MarkerSize = 8;
}
plot2.Title("Iris projected onto its first two principal components");
plot2.ShowLegend();
plot2.GetImageHtml(850, 600)